# 02 — News Cleaning and FinBERT Sentiment

## Purpose
Create one clean row per unique Alpaca article, then attach article-level FinBERT sentiment.

## Grain
**One row per unique article.** Do not expand the `symbols` list here. Ticker expansion happens in `02_news_ticker_mapping.ipynb`.

## Input
The raw Alpaca CSV in `data/raw/`.

## Output
`data/processed/news_articles_sentiment.csv`

## FinBERT columns
- `finbert_label`: `positive`, `negative` or `neutral` (highest-probability class).
- `finbert_positive`, `finbert_negative`, `finbert_neutral`: the three class probabilities.
- `finbert_score`: `P(positive) - P(negative)`, range -1 to +1.

Saving all three probabilities means either scoring convention can be recomputed later without re-running the model.

## How this notebook is run
Run the cells top to bottom. The FinBERT step works in chunks and saves each chunk to disk, so a crash or restart only costs the current chunk. Re-running the same cell resumes where it stopped.

In [2]:
import ast
import gc
import html
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline

# ---- Paths ----
RAW_PATH = Path("../../data/raw/alpaca_news_2021_2025_validated.csv")
OUTPUT_PATH = Path("../../data/processed/news_articles_sentiment.csv")
CKPT_ROOT = Path("../../data/processed")

# ---- Settings ----
KEEP_CONTENT = False   # 'content' is most of the file size and is not used by FinBERT
N_TEST = None          # e.g. 200 for a quick smoke test on a random sample; None = full run
CHUNK_SIZE = 2_000     # articles per checkpoint file
MAX_LENGTH = 128       # FinBERT token limit used for truncation

device = 0 if torch.cuda.is_available() else -1
print("FinBERT device:", "GPU" if device == 0 else "CPU")
torch.set_num_threads(4)  # only matters on CPU; adjust to your core count

d:\DEPI\MarketPulse Graduation Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FinBERT device: CPU


## 1. Load the raw CSV (memory-safe)

Most of the 327 MB is the `content` column, which FinBERT does not use. The file is read in chunks, all columns as text (so IDs are never turned into numbers), and `content` is skipped unless `KEEP_CONTENT = True`.

In [3]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {RAW_PATH.resolve()}. Update RAW_PATH to your raw Alpaca dataset."
    )

header = pd.read_csv(RAW_PATH, nrows=0).columns
use_cols = [c for c in header if KEEP_CONTENT or c != "content"]

parts = []
for part in pd.read_csv(RAW_PATH, usecols=use_cols, dtype=str, chunksize=20_000):
    parts.append(part)
raw = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

print(f"Loaded {len(raw):,} raw rows with {len(use_cols)} columns")

Loaded 79,549 raw rows with 10 columns


## 2. Check required source fields

Optional fields that are absent are created empty. The notebook stops if the article ID, timestamp or headline column is missing.

In [4]:
required = {"id", "created_at", "headline"}
missing_required = required - set(raw.columns)
if missing_required:
    raise ValueError(f"Missing required Alpaca columns: {sorted(missing_required)}")

for col in ["author", "content", "summary", "source", "updated_at", "url", "symbols"]:
    if col not in raw.columns:
        raw[col] = pd.NA

print("Required fields present.")

Required fields present.


## 3. Cleaning helpers

`clean_text` unescapes HTML entities, removes tags and collapses all whitespace (including newlines) to single spaces. `build_sentiment_text` joins headline and summary without stripping their own punctuation.

In [5]:
def clean_text(value):
    if value is None or pd.isna(value):
        return pd.NA
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)       # remove HTML tags
    text = re.sub(r"\s+", " ", text).strip()   # collapse all whitespace, incl. newlines
    return text if text else pd.NA


def parse_symbols(value):
    """Parse Alpaca symbols stored as a list, tuple, or list-like string."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        items = list(value)
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
            items = list(parsed) if isinstance(parsed, (list, tuple, set)) else [value]
        except (ValueError, SyntaxError):
            # Fallback only for a simple comma-separated string.
            items = value.split(",")
    else:
        return []
    return list(dict.fromkeys(
        str(item).strip().upper() for item in items
        if item is not None and str(item).strip()
    ))


def build_sentiment_text(title, summary):
    """Headline + summary; headline only if the summary is missing."""
    title = "" if pd.isna(title) else str(title).strip()
    summary = "" if pd.isna(summary) else str(summary).strip()
    if title and summary:
        sep = " " if title.endswith((".", "!", "?", ":")) else ". "
        return f"{title}{sep}{summary}"
    return title or summary or pd.NA

## 4. Standardize column names and clean the article-level dataset

`article_id` keeps Alpaca's original ID as a string. `published_at` and `updated_at` are parsed as UTC-aware timestamps. The original symbol text is kept as `symbols_raw`, and the parsed list in `symbols` is used by the mapping notebook.

In [6]:
articles = raw.rename(columns={
    "id": "article_id",
    "created_at": "published_at",
    "headline": "title",
}).copy()
del raw
gc.collect()

articles["article_id"] = articles["article_id"].astype("string").str.strip()
articles["published_at"] = pd.to_datetime(articles["published_at"], utc=True, errors="coerce")
articles["updated_at"] = pd.to_datetime(articles["updated_at"], utc=True, errors="coerce")

for col in ["title", "summary", "content", "author", "source", "url"]:
    articles[col] = articles[col].apply(clean_text)

articles["symbols_raw"] = articles["symbols"].astype("string")
articles["symbols"] = articles["symbols"].apply(parse_symbols)

# Rows without an original article ID cannot be reliably deduplicated or linked.
missing_id_count = int((articles["article_id"].isna() | articles["article_id"].str.len().fillna(0).eq(0)).sum())
print("Rows with missing article ID (dropped):", missing_id_count)
articles = articles[
    articles["article_id"].notna() & articles["article_id"].str.len().gt(0)
].copy()

# Keep rows missing a headline or timestamp, but flag them instead of inventing values.
articles["quality_missing_title"] = articles["title"].isna()
articles["quality_missing_published_at"] = articles["published_at"].isna()
print("Missing title:", int(articles["quality_missing_title"].sum()))
print("Missing published_at:", int(articles["quality_missing_published_at"].sum()))

Rows with missing article ID (dropped): 0
Missing title: 0
Missing published_at: 0


## 5. Deduplicate by original article ID

If an ID occurs more than once, the most recently updated version is kept. The original ID is the deduplication key, not `(ticker, title, timestamp)`.

In [7]:
duplicate_mask = articles["article_id"].duplicated(keep=False)
duplicate_rows = articles.loc[duplicate_mask].sort_values(["article_id", "updated_at"])
print("Rows in duplicate-ID groups:", len(duplicate_rows))
display(duplicate_rows[["article_id", "published_at", "updated_at", "title", "url"]].head(30))

articles = articles.sort_values(
    ["article_id", "updated_at"], na_position="first"
).drop_duplicates(subset=["article_id"], keep="last")

if articles["article_id"].duplicated().any():
    raise AssertionError("article_id is not unique after deduplication.")

print(f"Unique articles after cleaning: {len(articles):,}")

Rows in duplicate-ID groups: 0


,article_id,published_at,updated_at,title,url


Unique articles after cleaning: 79,549


## 6. Coverage check

The collection logs and the saved CSV disagreed before, so confirm the date range and monthly coverage of *this* file before trusting it. Nothing is filtered here; this only reports.

In [8]:
ts = articles["published_at"].dropna()
print("Earliest article:", ts.min())
print("Latest article:  ", ts.max())

start = pd.Timestamp("2021-01-01", tz="UTC")
end = pd.Timestamp("2026-01-01", tz="UTC")
print("Published before 2021:", int((ts < start).sum()))
print("Published after 2025: ", int((ts >= end).sum()))

monthly = (
    ts.dt.tz_localize(None).dt.to_period("M").value_counts().sort_index()
    .reindex(pd.period_range("2021-01", "2025-12", freq="M"), fill_value=0)
)
empty_months = monthly[monthly.eq(0)]
print(f"Months with zero articles: {len(empty_months)} of {len(monthly)}")
if len(empty_months):
    print([str(m) for m in empty_months.index])
display(monthly.describe())

Earliest article: 2021-01-02 13:28:07+00:00
Latest article:   2025-12-31 22:30:46+00:00
Published before 2021: 0
Published after 2025:  0
Months with zero articles: 0 of 60


count      60.000000
mean     1325.816667
std       189.084726
min       902.000000
25%      1211.000000
50%      1310.000000
75%      1458.250000
max      1717.000000
Name: count, dtype: float64

## 7. Prepare the text that FinBERT will score

Input is headline plus summary, or the headline alone when there is no summary. Text longer than `MAX_LENGTH` tokens is truncated by the tokenizer; document this setting in the project notes.

In [9]:
articles["sentiment_text"] = [
    build_sentiment_text(t, s) for t, s in zip(articles["title"], articles["summary"])
]

print("Rows with usable sentiment text:", int(articles["sentiment_text"].notna().sum()))
print("Texts still containing newlines:", int(articles["sentiment_text"].str.contains("\n", na=False).sum()))
display(articles[["article_id", "title", "summary", "sentiment_text"]].head())

Rows with usable sentiment text: 79549
Texts still containing newlines: 0


,article_id,title,summary,sentiment_text
35331,10903978,This Day In Market History: Warren Buffett Buy...,"Each day, Benzinga will take a look back at a ...",This Day In Market History: Warren Buffett Buy...
14438,10978750,This Day In Market History: Jobs And Wozniak I...,"Each day, Benzinga takes a look back at a nota...",This Day In Market History: Jobs And Wozniak I...
36340,11007650,This Day In Market History: Apple Introduces T...,"Each day, Benzinga takes a look back at a nota...",This Day In Market History: Apple Introduces T...
14723,11013619,This Day In Market History: Rockefeller Founds...,"Each day, Benzinga takes a look back at a nota...",This Day In Market History: Rockefeller Founds...
868,11093651,This Day In Market History: US Treasury Introd...,"Each day, Benzinga takes a look back at a nota...",This Day In Market History: US Treasury Introd...


## 8. Load FinBERT

In [10]:
finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    device=device,
)
print("FinBERT loaded.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 434.87it/s]


FinBERT loaded.


## 9. FinBERT inference (chunked and resumable)

Each chunk is saved to `data/processed/finbert_chunks/` keyed by `article_id`. If the kernel dies, restart it, run the cells above again, and re-run the cell below: finished chunks are skipped.

Texts are sorted by length so each batch has similar token counts (faster, less padding).

**Delete the `finbert_chunks` folder if you change the text-preparation step**, otherwise old chunks built from different text would be reused.

Set `N_TEST = 200` in the first cell for a smoke test on a random sample (uses a separate `finbert_chunks_test` folder). The export cell refuses to run while `N_TEST` is set.

In [11]:
texts = articles.set_index("article_id")["sentiment_text"].dropna().astype(str).str.strip()
texts = texts[texts.ne("")]
if N_TEST:
    texts = texts.sample(min(N_TEST, len(texts)), random_state=42)

order = texts.str.len().sort_values(kind="stable").index

CKPT_DIR = CKPT_ROOT / ("finbert_chunks_test" if N_TEST else "finbert_chunks")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
batch_size = 32 if device == 0 else 16
print(f"Scoring {len(order):,} articles in chunks of {CHUNK_SIZE:,} (batch size {batch_size})")

for start in tqdm(range(0, len(order), CHUNK_SIZE), desc="FinBERT chunks"):
    ckpt = CKPT_DIR / f"chunk_{start:06d}.csv"
    if ckpt.exists():
        continue

    ids = order[start:start + CHUNK_SIZE]
    with torch.inference_mode():
        results = finbert(
            texts.loc[ids].tolist(),
            batch_size=batch_size,
            truncation=True,
            max_length=MAX_LENGTH,
            top_k=None,
        )

    rows = []
    for article_id, class_results in zip(ids, results):
        p = {item["label"].lower(): float(item["score"]) for item in class_results}
        rows.append({
            "article_id": article_id,
            "finbert_label": max(p, key=p.get),
            "finbert_positive": p.get("positive", 0.0),
            "finbert_negative": p.get("negative", 0.0),
            "finbert_neutral": p.get("neutral", 0.0),
            "finbert_score": p.get("positive", 0.0) - p.get("negative", 0.0),
        })

    tmp = ckpt.with_suffix(".tmp")           # write to a temp file, then rename,
    pd.DataFrame(rows).to_csv(tmp, index=False)  # so a crash never leaves a half-written chunk
    tmp.replace(ckpt)

    del results, rows
    gc.collect()

print("All chunks done.")

Scoring 79,549 articles in chunks of 2,000 (batch size 16)


FinBERT chunks: 100%|██████████| 40/40 [11:25<00:00, 17.13s/it]

All chunks done.


## 10. Merge chunk results back into the article table

In [12]:
files = sorted(CKPT_DIR.glob("chunk_*.csv"))
scored = pd.concat(
    (pd.read_csv(f, dtype={"article_id": str}) for f in files),
    ignore_index=True,
)
if not scored["article_id"].is_unique:
    raise AssertionError("Duplicate article_id values across FinBERT chunks. Delete the chunk folder and re-run.")

not_scored = set(texts.index) - set(scored["article_id"])
if not_scored:
    raise RuntimeError(f"{len(not_scored):,} articles are not scored yet. Re-run the inference cell.")

new_cols = [c for c in scored.columns if c != "article_id"]
articles = articles.drop(columns=[c for c in new_cols if c in articles.columns]).merge(
    scored, on="article_id", how="left"
)
print(f"Merged scores for {scored['article_id'].nunique():,} articles")

del finbert, scored
gc.collect()

Merged scores for 79,549 articles


0

## 11. Validate sentiment outputs

Stops if the labels, scores or probabilities look wrong, so a dataset that merely looks complete is never exported.

In [13]:
required_sentiment_cols = {"finbert_label", "finbert_score"}
missing_sentiment_cols = required_sentiment_cols - set(articles.columns)
if missing_sentiment_cols:
    raise RuntimeError(f"FinBERT output columns are missing: {sorted(missing_sentiment_cols)}")

articles["finbert_label"] = articles["finbert_label"].astype("string").str.strip().str.lower()
articles["finbert_score"] = pd.to_numeric(articles["finbert_score"], errors="coerce")

valid_labels = {"positive", "negative", "neutral"}
unexpected_labels = set(articles["finbert_label"].dropna().unique()) - valid_labels
if unexpected_labels:
    raise ValueError(f"Unexpected labels found; map them consistently: {unexpected_labels}")

has_text = articles["sentiment_text"].notna()
unscored = int((has_text & articles["finbert_label"].isna()).sum())
print("Articles with text but no label:", unscored)
print("Articles without usable text (left unscored):", int((~has_text).sum()))
if unscored and not N_TEST:
    raise RuntimeError("Some articles with text were not scored.")

prob_cols = ["finbert_positive", "finbert_negative", "finbert_neutral"]
scored_rows = articles["finbert_label"].notna()
prob_error = (articles.loc[scored_rows, prob_cols].sum(axis=1) - 1).abs().max()
print("Max |sum of probabilities - 1|:", prob_error)

outside_range = int(articles["finbert_score"].dropna().abs().gt(1).sum())
if outside_range:
    raise ValueError(f"{outside_range} sentiment scores fall outside [-1, 1].")

display(articles["finbert_label"].value_counts(dropna=False).to_frame("article_count"))
display(articles["finbert_score"].describe())

Articles with text but no label: 0
Articles without usable text (left unscored): 0
Max |sum of probabilities - 1|: 1.3783574104309082e-07


,article_count
finbert_label,
neutral,35828
positive,23748
negative,19973


count    79549.000000
mean         0.054385
std          0.608743
min         -0.969381
25%         -0.321903
50%          0.052574
75%          0.588927
max          0.944173
Name: finbert_score, dtype: float64

## 12. Export one row per unique article

The list-valued `symbols` column is saved as JSON so the mapping notebook can parse it reliably. `sentiment_text` and the quality flags are kept for reproducibility and can be dropped from the warehouse table.

In [14]:
if N_TEST:
    raise RuntimeError("N_TEST is set, so this was only a smoke test. Set N_TEST = None and re-run from the inference cell.")

export_columns = [
    "article_id", "published_at", "updated_at", "title", "summary", "content",
    "author", "source", "url", "symbols_raw", "symbols",
    "finbert_label", "finbert_positive", "finbert_negative", "finbert_neutral", "finbert_score",
    "sentiment_text", "quality_missing_title", "quality_missing_published_at",
]
export_columns = [c for c in export_columns if c in articles.columns and (KEEP_CONTENT or c != "content")]

article_export = articles[export_columns].copy()
article_export["symbols"] = article_export["symbols"].apply(
    lambda value: json.dumps(value, ensure_ascii=False)
)
assert article_export["article_id"].is_unique

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
article_export.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(article_export):,} unique article rows to {OUTPUT_PATH.resolve()}")

Saved 79,549 unique article rows to D:\DEPI\MarketPulse Graduation Project\data\processed\news_articles_sentiment.csv


## Completion checklist

- [ ] No raw file was overwritten.
- [ ] `article_id` is preserved and unique.
- [ ] Timestamps are parsed as UTC-aware timestamps.
- [ ] Coverage check reviewed (date range, empty months).
- [ ] Missing titles/timestamps were counted and reviewed.
- [ ] FinBERT was run once per unique article.
- [ ] Score definition documented: `P(positive) - P(negative)`; text = headline + summary, truncated to 128 tokens.
- [ ] Output contains one row per unique article.
- [ ] FinBERT outputs are real model results, not placeholders.